# Lab 6 · MuJoCo 물리 시뮬레이션 픽앤플레이스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djfksjd/robot_study/blob/main/notebooks/03_mujoco_pick_place.ipynb)

웹 시뮬레이터는 '조건이 맞으면 붙이는' 운동학 시뮬레이션이다. 여기서는 마찰·접촉·중력이 있는 물리 엔진에서
**같은 JSON 계획**이 실제로 성공하는지 본다.

In [ ]:
# 0) 저장소 준비 — Colab에서는 처음 한 번 clone, 로컬에서는 저장소 루트에서 실행하면 건너뛴다
import os, sys
if not os.path.exists("model/scara_params.json"):
    if not os.path.exists("robot_study"):
        !git clone -q https://github.com/djfksjd/robot_study.git
    %cd robot_study
sys.path.insert(0, "python")
print("작업 폴더:", os.getcwd())

In [ ]:
# Colab: 물리 엔진 설치 + 화면 없는 렌더링 설정
import os, sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mujoco", "mediapy"], check=True)
    os.environ["MUJOCO_GL"] = "egl"
import mujoco
print("MuJoCo", mujoco.__version__)

In [ ]:
sys.path.insert(0, "sim/mujoco")
import importlib, pick_place
importlib.reload(pick_place)

plan = {"tasks": [
    {"action": "pick", "object": "red"},   {"action": "place", "target": "tray_A"},
    {"action": "pick", "object": "blue"},  {"action": "place", "target": "tray_B"},
    {"action": "pick", "object": "green"}, {"action": "place", "target": "tray_A"},
    {"action": "home"}]}

RENDER = "google.colab" in sys.modules  # 로컬에서는 영상 없이 물리만
sim = pick_place.Sim(video="out.mp4" if RENDER else None, fps=20)
try:
    sim.run(plan)
except pick_place.SafetyError as e:
    print("SAFETY STOP:", e)
print("\n".join(sim.log))
for n in ("red", "green", "blue"):
    print(n, sim.body_xyz(n).round(1))

In [ ]:
# 결과 판정: 목표 트레이 중심에서 5 mm 이내, 높이가 맞는가
import numpy as np
ok = (np.hypot(*(sim.body_xyz("red")[:2] - [60, 330])) < 5 and abs(sim.body_xyz("green")[2] - 45) < 3
      and np.hypot(*(sim.body_xyz("blue")[:2] - [60, -330])) < 5)
print("PASS" if ok else "FAIL")

In [ ]:
# 영상 (Colab)
if RENDER and sim.frames:
    import mediapy
    mediapy.show_video(sim.frames, fps=sim.fps)

## 실험 과제 (한 번에 하나만 바꾸고 결과를 기록)
1. `model/task_scene.json` 의 `grip_closed_mm` 를 28로 바꾸면? (쥐는 힘이 줄어든다)
2. `sim/mujoco/build_mjcf.py` 의 패드 마찰 `friction="1.5 ..."` 를 0.3으로 → `python3 sim/mujoco/build_mjcf.py` 로 다시 생성
3. 블록 질량 0.03 → 0.3 kg
기록 양식은 `seminar/lab_report_template.md`.